# Démonstration End-to-End : Analyse du Cancer du Sein en France

Ce notebook rassemble l'intégralité de la chaîne réelle de traitement des données du projet (`Hythus/dbt-bigquery-analyse-cancer-du-sein`) :

1. **Collecte API & Open Data (`sources_python/api_secu/API sécu.ipynb`)** :
   * **Assurance Maladie (Ameli)** : `effectifs` (`df_patients_sein`), `depenses` (`df_depenses_onco`), `referentiel-regions` & `referentiel-departements` via l'endpoint `/exports/json`.
   * **DREES (Morbidité hospitalière)** : archive ZIP `500_morbidite-hospitaliere` (`tableau2`) → `df_dms_onco` et `df_charge_onco` découpés en 13 tables par granularité (`France`, `Région`, `Département`).
   * **CépiDc (Inserm) & Ameli tous cancers (`extract_comparatif_onco.py`)** : mortalité régionale & par âge (`Tumeurs` / `Tumeurs malignes`, 2014–2024) et effectifs `Cancers` (`dept=999`, 2015–2024).
2. **Extraction & Nettoyage PDF des Centres de Radiologie (`sources_python/radiologie_pdf/`)** :
   * Parsing de `PDF France entière.pdf` via `src.extractor.extract_centers` (`extraction_centre_depistage.ipynb`).
   * Nettoyage (`cleaning_depistages.ipynb`) : exclusion de Monaco, résolution Corse `2A`/`2B` via `geo.api.gouv.fr/communes` + correction de `L'Île-Rousse`, normalisation Unicode `NFC` & `zfill(2)` → chargement BigQuery de `Data_brute.df_radiologie_final`.
3. **Architecture Médaillon BigQuery & dbt (`cancer-du-sein-513216`)** :
   * **Data Lake (`Data_brute`)** : 53 tables brutes consolidées (+ 18 tables `Data_clean` en lecture seule).
   * **Bronze / Staging (`stg_projet_cancer`)** : 26 vues dbt de standardisation, typage, dédoublonnage et clés SHA-256.
   * **Silver / Intermediate (`int_projet_cancer`)** : 8 tables métier croisées par problématique.
   * **Gold / Marts (`mart_projet_cancer`)** : modèle en étoile dbt composé de 2 dimensions (`dim_regions`, `dim_tranches_age`) et 6 tables de faits (`mart_depistage_centres_region`, `mart_carte_centres_ville`, `mart_correlation_depistage_prevalence_mortalite_region`, `mart_correlation_age_region_sein`, `mart_oncologie_depenses_charge_region`, `mart_comparatif_sein_vs_onco`).
4. **Qualité, GitHub & Power BI / DAX** : tests automatisés `dbt test`, suivi GitHub et reproduction des KPIs Power BI.

## 0. Configuration de l'environnement, des chemins et de BigQuery

**Ce que fait la cellule ci-dessous :**  
Charge les librairies Python (`requests`, `pandas`, `numpy`, `zipfile`, `unicodedata`, `google-cloud-bigquery`, `subprocess`), configure les chemins locaux vers les scripts copiés (`sources_python/api_secu`, `sources_python/radiologie_pdf`) et le projet dbt, puis initialise le client BigQuery sur `cancer-du-sein-513216`.

In [1]:
import json
import os
import re
import subprocess
import sys
import unicodedata
import zipfile
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
import requests
from google.cloud import bigquery

PROJECT_ID = "cancer-du-sein-513216"
DBT_PROJECT_DIR = Path(r"c:\dev\dbt-workspace\mon_projet")
DBT_EXE = Path(r"c:\dev\dbt-workspace\.venv\Scripts\dbt.exe")
API_DIR = DBT_PROJECT_DIR / "sources_python" / "api_secu"
RAD_DIR = DBT_PROJECT_DIR / "sources_python" / "radiologie_pdf"
RAW_DIR = API_DIR / "out" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_CREDS = Path(r"C:\Users\Deval\Documents\data bird\Projet\cancer-du-sein-513216-f2aa37ec2318.json")
if "GOOGLE_APPLICATION_CREDENTIALS" not in os.environ and DEFAULT_CREDS.exists():
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(DEFAULT_CREDS)

client = bigquery.Client(project=PROJECT_ID)
print(f"Connecté à BigQuery : {client.project}")
print(f"Dossier API         : {API_DIR}")
print(f"Dossier Radiologie  : {RAD_DIR}")

Connecté à BigQuery : cancer-du-sein-513216
Dossier API         : c:\dev\dbt-workspace\mon_projet\sources_python\api_secu
Dossier Radiologie  : c:\dev\dbt-workspace\mon_projet\sources_python\radiologie_pdf


## 1.1. Fonctions d'extraction API Ameli (`/exports/json`) et codification géographique (`API sécu.ipynb`)

**Ce que fait la cellule ci-dessous :**  
Implémente `export_ameli(dataset, where)` sur l'endpoint d'export complet de `data.ameli.fr` (sans limite de 10 000 lignes) et `niveau_geo_ameli(df)` (`region == '99'` → France, `dept == '999'` → Région, sinon Département), puis charge les référentiels officiels `referentiel-regions` et `referentiel-departements`.

In [2]:
AMELI_URL = "https://data.ameli.fr/api/explore/v2.1/catalog/datasets/{dataset}/exports/json"


def export_ameli(dataset, where):
    """Récupère tous les enregistrements d'un dataset Ameli (endpoint d'export), filtrés en ODSQL."""
    response = requests.get(
        AMELI_URL.format(dataset=dataset),
        headers={"Accept": "application/json"},
        params={"where": where},
        timeout=120,
    )
    if response.status_code == 200:
        records = response.json()
        print(f"{dataset} : {len(records):,} enregistrements récupérés")
        return pd.DataFrame(records)
    print(f"Erreur HTTP {response.status_code} : {response.text}")
    return pd.DataFrame()


def niveau_geo_ameli(df):
    """Codification Ameli : region = '99' -> France ; dept = '999' -> total région ; sinon département."""
    df["niveau"] = np.select(
        [df["region"] == "99", df["dept"] == "999"],
        ["France", "Région"],
        default="Département",
    )
    df["code_zone"] = np.select(
        [df["niveau"] == "France", df["niveau"] == "Région"],
        ["FR", df["region"]],
        default=df["dept"],
    )
    return df


ref_regions = export_ameli("referentiel-regions", "reg_code IS NOT NULL")
ref_depts = export_ameli("referentiel-departements", "dep_code IS NOT NULL")

nom_region = dict(zip(ref_regions["reg_code"], ref_regions["reg_name"]))
nom_dept = dict(zip(ref_depts["dep_code"], ref_depts["dep_name"]))
region_du_dept = dict(zip(ref_depts["dep_code"], ref_depts["reg_code"]))
print(f"Référentiels chargés : {len(nom_region)} régions, {len(nom_dept)} départements")

referentiel-regions : 27 enregistrements récupérés
referentiel-departements : 110 enregistrements récupérés
Référentiels chargés : 27 régions, 110 départements


## 1.2. DF 1 : Extraction et nettoyage de `df_patients_sein` (Ameli `effectifs`)

**Ce que fait la cellule ci-dessous :**  
Extrait depuis Ameli `effectifs` toutes les lignes `patho_niv2 = "Cancer du sein de la femme"` (2015–2024), convertit `annee` en entier, remplace `patho_niv3` vide par `'Ensemble'`, applique `niveau_geo_ameli()`, renomme les colonnes métier (`nb_patients`, `population`, `prevalence_pct`) et affiche l'évolution nationale totale (`France`, `tsage`, `sexe = '9'`, `Ensemble`).

In [3]:
df_patients_sein = export_ameli("effectifs", 'patho_niv2 = "Cancer du sein de la femme"')

df_patients_sein["annee"] = pd.to_datetime(df_patients_sein["annee"]).dt.year
df_patients_sein["patho_niv3"] = df_patients_sein["patho_niv3"].fillna("Ensemble")
df_patients_sein = niveau_geo_ameli(df_patients_sein)

df_patients_sein = (
    df_patients_sein.rename(
        columns={
            "region": "code_region",
            "ntop": "nb_patients",
            "npop": "population",
            "prev": "prevalence_pct",
        }
    )[
        [
            "annee", "niveau", "code_zone", "code_region", "dept",
            "patho_niv2", "patho_niv3", "top", "cla_age_5", "libelle_classe_age",
            "sexe", "libelle_sexe", "nb_patients", "population", "prevalence_pct",
        ]
    ]
    .sort_values(["annee", "niveau", "code_zone", "top", "sexe", "cla_age_5"])
    .reset_index(drop=True)
)

print(f"df_patients_sein : {df_patients_sein.shape[0]:,} lignes x {df_patients_sein.shape[1]} colonnes")
display(
    df_patients_sein[
        (df_patients_sein["niveau"] == "France")
        & (df_patients_sein["cla_age_5"] == "tsage")
        & (df_patients_sein["sexe"] == "9")
        & (df_patients_sein["patho_niv3"] == "Ensemble")
    ][["annee", "nb_patients", "population", "prevalence_pct"]]
)

effectifs : 151,200 enregistrements récupérés


df_patients_sein : 151,200 lignes x 15 colonnes


,annee,nb_patients,population,prevalence_pct
12809,2015,685350.0,33622410,2.038
27929,2016,679610.0,33847120,2.008
43049,2017,695750.0,33988170,2.047
58169,2018,714910.0,34164050,2.093
73289,2019,727280.0,34379350,2.115
88409,2020,729310.0,34401380,2.120
103529,2021,731890.0,35242500,2.077
118649,2022,726650.0,35442250,2.050
133769,2023,727060.0,35047220,2.075
148889,2024,728980.0,35171580,2.073


## 1.3. Découpage de `df_patients_sein` par échelle géographique (`df_sein_france`, `df_sein_region`, `df_sein_dept`)

**Ce que fait la cellule ci-dessous :**  
Découpe `df_patients_sein` en trois DataFrames selon les totaux publiés par Ameli (sans sommer les départements afin d'éviter la perte des petits effectifs soumis au secret statistique) et associe les libellés officiels `nom_region` et `nom_dept`.

In [4]:
base = df_patients_sein.reset_index() if "niveau" in df_patients_sein.index.names else df_patients_sein.copy()
colonnes_geo = ["niveau", "code_zone", "code_region", "dept"]
colonnes_donnees = [c for c in base.columns if c not in colonnes_geo]


def extraire(niveau, colonnes_geo_a_garder):
    df = base[base["niveau"] == niveau][colonnes_geo_a_garder + colonnes_donnees]
    if colonnes_geo_a_garder:
        df = df.sort_values(colonnes_geo_a_garder, kind="stable")
    return df.reset_index(drop=True)


df_sein_france = extraire("France", [])
df_sein_region = extraire("Région", ["code_region"])
df_sein_dept = extraire("Département", ["code_region", "dept"])

df_sein_france.insert(0, "territoire", "France")
for df in (df_sein_region, df_sein_dept):
    df.insert(0, "nom_region", df.pop("code_region").map(nom_region))
df_sein_dept.insert(df_sein_dept.columns.get_loc("dept") + 1, "nom_dept", df_sein_dept["dept"].map(nom_dept))

for nom, d in [("df_sein_france", df_sein_france), ("df_sein_region", df_sein_region), ("df_sein_dept", df_sein_dept)]:
    print(f"{nom} : {d.shape[0]:,} lignes x {d.shape[1]} colonnes")

display(df_sein_region.head(5))

df_sein_france : 1,260 lignes x 12 colonnes
df_sein_region : 22,680 lignes x 12 colonnes
df_sein_dept : 127,260 lignes x 14 colonnes


,nom_region,annee,patho_niv2,patho_niv3,top,cla_age_5,libelle_classe_age,sexe,libelle_sexe,nb_patients,population,prevalence_pct
0,Guadeloupe,2015,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,00-04,de 0 à 4 ans,2,femmes,NaN,10580,NaN
1,Guadeloupe,2015,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,05-09,de 5 à 9 ans,2,femmes,NaN,12320,NaN
2,Guadeloupe,2015,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,10-14,de 10 à 14 ans,2,femmes,NaN,13070,NaN
3,Guadeloupe,2015,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,15-19,de 15 à 19 ans,2,femmes,NaN,13290,NaN
4,Guadeloupe,2015,Cancer du sein de la femme,Cancer du sein de la femme actif,CAN_SEI_ACT,20-24,de 20 à 24 ans,2,femmes,NaN,10280,NaN


## 1.4. DF 2 : Extraction et regroupement de `df_depenses_onco` (Ameli `depenses`)

**Ce que fait la cellule ci-dessous :**  
Extrait depuis Ameli `depenses` toutes les dépenses liées aux cancers (`patho_niv1 = "Cancers"`), regroupe les 31 postes bruts `dep_niv_2` en catégories lisibles via `TYPES_DEPENSE`, identifie le secteur (`Public`, `Privé`, `Non précisé`) et affiche un tableau comparatif en M€ sur la dernière année disponible.

In [5]:
df_depenses_onco = export_ameli("depenses", 'patho_niv1 = "Cancers"')

df_depenses_onco["annee"] = pd.to_datetime(df_depenses_onco["annee"]).dt.year
df_depenses_onco["patho_niv2"] = df_depenses_onco["patho_niv2"].fillna("Tous cancers")
df_depenses_onco["patho_niv3"] = df_depenses_onco["patho_niv3"].fillna("Ensemble")

TYPES_DEPENSE = {
    "Hospitalisations séjour MCO secteur public remboursées": "Hôpital - séjours MCO",
    "Hospitalisations séjour MCO secteur privé remboursées": "Hôpital - séjours MCO",
    "Hospitalisations liste en sus MCO secteur public remboursées": "Hôpital - liste en sus MCO",
    "Hospitalisations liste en sus MCO secteur privé remboursées": "Hôpital - liste en sus MCO",
    "Actes et consultations externes MCO secteur public remboursés": "Hôpital - actes et consultations externes",
    "Hospitalisations en SSR secteur public remboursées": "SSR (soins de suite et de réadaptation)",
    "Hospitalisations en SSR secteur privé remboursées": "SSR (soins de suite et de réadaptation)",
    "Hospitalisations en HAD secteur public remboursées": "HAD (hospitalisation à domicile)",
    "Hospitalisations en HAD secteur privé remboursées": "HAD (hospitalisation à domicile)",
    "Hospitalisations en psychiatrie secteur public remboursées": "Psychiatrie",
    "Hospitalisations en psychiatrie secteur privé remboursées": "Psychiatrie",
    "Médicaments remboursés": "Ville - médicaments",
    "Biologie remboursée": "Ville - biologie",
    "Soins infirmiers remboursés": "Ville - soins infirmiers",
    "Soins de kinésithérapie remboursés": "Ville - kinésithérapie",
    "Soins d'autres paramédicaux remboursés": "Ville - autres paramédicaux",
    "Soins de généralistes remboursés": "Ville - généralistes",
    "Soins autres spécialistes remboursés": "Ville - spécialistes",
    "Soins de sages-femmes remboursés": "Ville - sages-femmes",
    "Soins dentaires remboursés": "Ville - dentaire",
    "Transports remboursés": "Ville - transports",
    "Autres produits de santé remboursés": "Ville - autres produits de santé",
    "Autres dépenses de soins de ville remboursés": "Ville - autres dépenses",
    "Indemnités journalières maladie et AT/MP remboursées": "Prestations - IJ maladie et AT/MP",
    "Indemnités journalières maternité et autres prestations remboursées": "Prestations - IJ maternité et autres",
    "Prestations d'invalidité remboursées": "Prestations - invalidité",
    "Rentes de victimes ATMP remboursées": "Prestations - rentes AT/MP",
    "Total hospitalisations (tous secteurs) remboursées": "TOTAL hôpital",
    "Total soins de ville remboursés": "TOTAL ville",
    "Total prestations en espèces remboursées": "TOTAL prestations en espèces",
    "Total des dépenses remboursées": "TOTAL général",
}
df_depenses_onco["type_depense"] = df_depenses_onco["dep_niv_2"].map(TYPES_DEPENSE)
df_depenses_onco["secteur"] = np.select(
    [
        df_depenses_onco["dep_niv_2"].str.contains("secteur public"),
        df_depenses_onco["dep_niv_2"].str.contains("secteur privé"),
    ],
    ["Public", "Privé"],
    default="Non précisé",
)

df_depenses_onco = (
    df_depenses_onco.rename(columns={"ntop": "nb_patients_pathologie"})[
        [
            "annee", "patho_niv2", "patho_niv3", "top", "dep_niv_1", "type_depense",
            "secteur", "dep_niv_2", "type_somme", "montant", "nb_patients_pathologie",
            "n_recourant_au_poste", "montant_moy",
        ]
    ]
    .sort_values(["annee", "top", "dep_niv_1", "type_somme", "dep_niv_2"])
    .reset_index(drop=True)
)

print(f"df_depenses_onco : {df_depenses_onco.shape[0]:,} lignes x {df_depenses_onco.shape[1]} colonnes")
detail = df_depenses_onco[
    (df_depenses_onco["annee"] == df_depenses_onco["annee"].max())
    & (df_depenses_onco["type_somme"] == "Partiel")
    & (df_depenses_onco["patho_niv3"] == "Ensemble")
]
display((detail.pivot_table(index="type_depense", columns="patho_niv2", values="montant", aggfunc="sum") / 1e6).round(1))

depenses : 4,960 enregistrements récupérés
df_depenses_onco : 4,960 lignes x 13 colonnes


patho_niv2,Autres cancers,Cancer bronchopulmonaire,Cancer colorectal,Cancer de la prostate,Cancer du sein de la femme,Tous cancers
type_depense,,,,,,
HAD (hospitalisation à domicile),526.2,109.0,77.2,40.0,114.3,866.6
Hôpital - actes et consultations externes,342.5,77.5,45.8,68.2,136.3,670.3
Hôpital - liste en sus MCO,4153.3,1269.0,239.9,159.0,1133.0,6954.2
Hôpital - séjours MCO,4714.5,883.8,910.3,569.4,1093.5,8171.5
Prestations - IJ maladie et AT/MP,277.5,53.8,45.0,31.1,261.0,668.3
Prestations - IJ maternité et autres,0.0,0.0,0.0,0.0,0.0,0.0
Prestations - invalidité,298.7,59.1,49.0,13.3,202.0,622.1
Prestations - rentes AT/MP,97.3,53.6,16.6,64.0,7.4,238.9
Psychiatrie,0.0,0.0,0.0,0.0,0.0,0.0


## 1.5. DF 3 & DF 4 : Morbidité hospitalière DREES (`df_mh` → `df_dms_onco` et `df_charge_onco`)

**Ce que fait la cellule ci-dessous :**  
Lit le tableau 2 de l'archive ZIP DREES (`morbidite_hospitaliere_tableaux_csv.zip`), harmonise la codification géographique avec Ameli, isole les tumeurs (`CODES_ONCO` `02000` à `02032`), calcule `nb_sejours_24h_plus`, `part_ambulatoire` et `nb_journees_estimees`, puis découpe `df_dms_onco` et `df_charge_onco` par échelle géographique via `decouper_par_niveau()`.

In [6]:
mh_url = (
    "https://data.drees.solidarites-sante.gouv.fr/api/v2/catalog/datasets/"
    "500_morbidite-hospitaliere/attachments/morbidite_hospitaliere_tableaux_format_csv_zip"
)
zip_path = RAW_DIR / "morbidite_hospitaliere_tableaux_csv.zip"

if not zip_path.exists():
    response = requests.get(mh_url, stream=True, timeout=600)
    response.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
    print(f"Archive téléchargée : {zip_path}")
else:
    print(f"Archive déjà présente localement : {zip_path.name}")

with zipfile.ZipFile(zip_path) as z:
    nom_t2 = next(n for n in z.namelist() if "tableau2" in n.lower())
    with z.open(nom_t2) as f:
        df_mh = pd.read_csv(
            f,
            sep=";",
            encoding="utf-8-sig",
            na_values=["ND"],
            dtype={"ANNEE": int, "Niveau": str, "PATHOLOGIE": str, "ZONE": str},
        )

df_mh[["code_patho", "libelle_patho"]] = df_mh["PATHOLOGIE"].str.split("-", n=1, expand=True)
df_mh["niveau"] = df_mh["Niveau"].map({"France": "France", "Régions": "Région", "Départements": "Département"})
df_mh[["code_zone", "libelle_zone"]] = df_mh["ZONE"].str.split(" - ", n=1, expand=True)
df_mh.loc[df_mh["code_zone"] == "3", ["code_zone", "libelle_zone"]] = ["FR", "France entière"]
df_mh.loc[df_mh["code_zone"] == "1", ["code_zone", "libelle_zone"]] = ["FRM", "France métropolitaine"]

df_mh = df_mh.rename(
    columns={
        "ANNEE": "annee",
        "<24h hospitalisations programmées": "sej_moins_24h_programmes",
        "<24h autres hospitalisations": "sej_moins_24h_autres",
        "<24h ensemble des hospitalisations": "nb_sejours_moins_24h",
        "1 jour": "sej_1j", "2 jours": "sej_2j", "3 jours": "sej_3j", "4 jours": "sej_4j", "5 jours": "sej_5j",
        "6 jours": "sej_6j", "7 jours": "sej_7j", "8 jours": "sej_8j", "9 jours": "sej_9j",
        "10 à 19 jours": "sej_10_19j", "20 à 29 jours": "sej_20_29j", "30 jours et plus": "sej_30j_plus",
        "Ensemble des hospitalisations": "nb_sejours",
        "Durée moyenne de séjour (en jours)": "dms_jours",
    }
).drop(columns=["Niveau", "ZONE", "PATHOLOGIE", "Données"])

CODES_ONCO = sorted(c for c in df_mh["code_patho"].unique() if c.startswith("02"))
NIVEAU_PATHO = {
    "02000": "Chapitre (toutes tumeurs)",
    "02007": "Sous-total tumeurs malignes",
    "02025": "Sous-total tumeurs bénignes",
}
df_mh["niveau_patho"] = df_mh["code_patho"].map(NIVEAU_PATHO).fillna("Détail")

df_dms_onco = df_mh[df_mh["code_patho"].isin(CODES_ONCO)].copy()
df_dms_onco["nb_sejours_24h_plus"] = df_dms_onco["nb_sejours"] - df_dms_onco["nb_sejours_moins_24h"]
df_dms_onco = df_dms_onco[
    ["annee", "niveau", "code_zone", "libelle_zone", "code_patho", "libelle_patho", "niveau_patho", "nb_sejours_24h_plus", "dms_jours"]
].reset_index(drop=True)

df_charge_onco = df_mh[df_mh["code_patho"].isin(CODES_ONCO)].copy()
df_charge_onco["part_ambulatoire"] = df_charge_onco["nb_sejours_moins_24h"] / df_charge_onco["nb_sejours"]
df_charge_onco["nb_journees_estimees"] = (df_charge_onco["nb_sejours"] - df_charge_onco["nb_sejours_moins_24h"]) * df_charge_onco["dms_jours"]
colonnes_duree = [
    "sej_1j", "sej_2j", "sej_3j", "sej_4j", "sej_5j", "sej_6j", "sej_7j",
    "sej_8j", "sej_9j", "sej_10_19j", "sej_20_29j", "sej_30j_plus",
]
df_charge_onco = df_charge_onco[
    ["annee", "niveau", "code_zone", "libelle_zone", "code_patho", "libelle_patho", "niveau_patho",
     "nb_sejours", "nb_sejours_moins_24h", "sej_moins_24h_programmes", "sej_moins_24h_autres"]
    + colonnes_duree + ["part_ambulatoire", "nb_journees_estimees"]
].reset_index(drop=True)


def decouper_par_niveau(df):
    colonnes_geo = ["niveau", "code_zone", "libelle_zone", "code_region"]
    colonnes_donnees = [c for c in df.columns if c not in colonnes_geo]
    france = df[df["niveau"] == "France"].copy()
    france.insert(0, "territoire", france["code_zone"].map({"FR": "France", "FRM": "France métropolitaine"}))
    region = df[df["niveau"] == "Région"].sort_values("code_zone", kind="stable").copy()
    region.insert(0, "nom_region", region["code_zone"].map(nom_region))
    dept = df[df["niveau"] == "Département"].copy()
    dept["code_reg_tri"] = dept["code_zone"].map(region_du_dept)
    dept = dept.sort_values(["code_reg_tri", "code_zone"], kind="stable")
    dept.insert(0, "nom_region", dept["code_reg_tri"].map(nom_region))
    dept.insert(1, "dept", dept["code_zone"])
    dept.insert(2, "nom_dept", dept["code_zone"].map(nom_dept))
    return (
        france[["territoire"] + colonnes_donnees].reset_index(drop=True),
        region[["nom_region"] + colonnes_donnees].reset_index(drop=True),
        dept[["nom_region", "dept", "nom_dept"] + colonnes_donnees].reset_index(drop=True),
    )


df_dms_france, df_dms_region, df_dms_dept = decouper_par_niveau(df_dms_onco)
df_charge_france, df_charge_region, df_charge_dept = decouper_par_niveau(df_charge_onco)

print(f"df_dms_onco    : {df_dms_onco.shape[0]:,} lignes ({len(df_dms_region):,} région)")
print(f"df_charge_onco : {df_charge_onco.shape[0]:,} lignes ({len(df_charge_region):,} région)")
display(df_charge_france[(df_charge_france["territoire"] == "France") & (df_charge_france["code_patho"] == "02017")][["annee", "nb_sejours", "nb_sejours_moins_24h", "part_ambulatoire", "nb_journees_estimees"]])

Archive déjà présente localement : morbidite_hospitaliere_tableaux_csv.zip


df_dms_onco    : 22,869 lignes (3,402 région)
df_charge_onco : 22,869 lignes (3,402 région)


,annee,nb_sejours,nb_sejours_moins_24h,part_ambulatoire,nb_journees_estimees
23,2018,77505.0,26925.0,0.347397,199285.20
77,2019,77009.0,29196.0,0.379125,185992.57
131,2020,73370.0,30279.0,0.412689,158574.88
185,2021,79853.0,34732.0,0.434949,162435.60
239,2022,77602.0,34465.0,0.444125,154430.46
293,2023,80823.0,36945.0,0.457110,156205.68
347,2024,81303.0,38237.0,0.470302,150731.00


## 2.1. Extraction des centres de radiologie depuis le PDF (`extraction_centre_depistage.ipynb`)

**Ce que fait la cellule ci-dessous :**  
Exécute le module `src.extractor.extract_centers` directement sur le fichier `PDF France entière.pdf` copié dans `sources_python/radiologie_pdf/` pour parser les annuaires régionaux et produire `df_raw` et `df_clean`.

In [7]:
if str(RAD_DIR) not in sys.path:
    sys.path.insert(0, str(RAD_DIR))

from src.extractor import extract_centers

pdf_path = next(RAD_DIR.glob("PDF France*.pdf"))
df_raw, df_clean = extract_centers(pdf_path, output=RAD_DIR / "output")

print(f"PDF parsé : {pdf_path.name}")
print(f"df_raw    : {df_raw.shape[0]:,} lignes x {df_raw.shape[1]} colonnes")
print(f"df_clean  : {df_clean.shape[0]:,} lignes x {df_clean.shape[1]} colonnes")
display(df_raw[["code_departement", "departement", "nom_centre", "ville", "page_source"]].head(10))

PDF parsé : PDF France entière.pdf
df_raw    : 1,657 lignes x 20 colonnes
df_clean  : 1,613 lignes x 4 colonnes


,code_departement,departement,nom_centre,ville,page_source
0,01,Ain,CENTRE D'IMAGERIE MEDICALE - CABINET D'AMBERIE...,AMBERIEU EN BUGEY,1
1,01,Ain,CH BUGEY SUD - SERVICE DE RADIOLOGIE,BELLEY,1
2,01,Ain,CENTRE DE MAMMOGRAPHIE DE BOURG EN BRESSE,BOURG EN BRESSE,1
3,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE L...,CHATILLON/CHALARONNE,1
4,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE M...,MEXIMIEUX,1
5,01,Ain,CABINET DE RADIOLOGIE D'OYONNAX,OYONNAX,1
6,01,Ain,CABINET DE RADIOLOGIE IMAGERIE MEDICALE DU PAY...,ST GENIS POUILLY,1
7,01,Ain,IMAGERIE CALADOISE SITE DE TREVOUX,TREVOUX,1
8,02,Aisne,IMAGERIE MEDICALE CASTEL,CHATEAU THIERRY,2
9,02,Aisne,CABINET DE RADIOLOGIE IMAGERIE MEDICALE,CHAUNY,2


## 2.2. Nettoyage des centres : mapping INSEE des 18 régions et exclusion de Monaco (`cleaning_depistages.ipynb`)

**Ce que fait la cellule ci-dessous :**  
Normalise `code_departement` (`1` / `1.0` → `"01"`, conservation de `"2A"`/`"2B"`), associe `Code_region` et `Region` via le dictionnaire officiel INSEE des 18 régions, exclut les établissements de Monaco (`99138`, `MC`, `"monaco"`) et supprime les doublons initiaux.

In [8]:
regions = {
    "01": ("Guadeloupe", "971"),
    "02": ("Martinique", "972"),
    "03": ("Guyane", "973"),
    "04": ("La Réunion", "974"),
    "06": ("Mayotte", "976"),
    "11": ("Île-de-France", "75 77 78 91 92 93 94 95"),
    "24": ("Centre-Val de Loire", "18 28 36 37 41 45"),
    "27": ("Bourgogne-Franche-Comté", "21 25 39 58 70 71 89 90"),
    "28": ("Normandie", "14 27 50 61 76"),
    "32": ("Hauts-de-France", "02 59 60 62 80"),
    "44": ("Grand Est", "08 10 51 52 54 55 57 67 68 88"),
    "52": ("Pays de la Loire", "44 49 53 72 85"),
    "53": ("Bretagne", "22 29 35 56"),
    "75": ("Nouvelle-Aquitaine", "16 17 19 23 24 33 40 47 64 79 86 87"),
    "76": ("Occitanie", "09 11 12 30 31 32 34 46 48 65 66 81 82"),
    "84": ("Auvergne-Rhône-Alpes", "01 03 07 15 26 38 42 43 63 69 73 74"),
    "93": ("Provence-Alpes-Côte d'Azur", "04 05 06 13 83 84"),
    "94": ("Corse", "2A 2B"),
}

departement_region = {
    dep: code_region
    for code_region, (_, departements) in regions.items()
    for dep in departements.split()
}

df_radiologie = df_raw[["code_departement", "departement", "nom_centre", "ville"]].copy()
df_radiologie["code_departement"] = (
    df_radiologie["code_departement"]
    .astype("string").str.strip().str.upper().str.replace(r"\.0+$", "", regex=True).str.zfill(2)
)
df_radiologie["Code_region"] = df_radiologie["code_departement"].map(departement_region).astype("string")
df_radiologie["Region"] = df_radiologie["Code_region"].map({code: nom for code, (nom, _) in regions.items()})
df_radiologie = df_radiologie[["Code_region", "Region", "code_departement", "departement", "nom_centre", "ville"]]

nb_initial = len(df_radiologie)
monaco = df_radiologie["code_departement"].astype("string").str.strip().str.upper().isin(["99138", "MC"])
for colonne in ["departement", "ville", "nom_centre"]:
    monaco |= df_radiologie[colonne].astype("string").str.contains(r"\bmonaco\b", case=False, na=False)

nb_monaco = int(monaco.sum())
df_radiologie = df_radiologie.loc[~monaco].drop_duplicates().replace(r"^\s*$", pd.NA, regex=True).reset_index(drop=True)

print(f"Lignes initiales : {nb_initial} | Monaco exclues : {nb_monaco} | Restantes : {len(df_radiologie)}")
display(df_radiologie.head(5))

Lignes initiales : 1657 | Monaco exclues : 1 | Restantes : 1571


,Code_region,Region,code_departement,departement,nom_centre,ville
0,84,Auvergne-Rhône-Alpes,01,Ain,CENTRE D'IMAGERIE MEDICALE - CABINET D'AMBERIE...,AMBERIEU EN BUGEY
1,84,Auvergne-Rhône-Alpes,01,Ain,CH BUGEY SUD - SERVICE DE RADIOLOGIE,BELLEY
2,84,Auvergne-Rhône-Alpes,01,Ain,CENTRE DE MAMMOGRAPHIE DE BOURG EN BRESSE,BOURG EN BRESSE
3,84,Auvergne-Rhône-Alpes,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE L...,CHATILLON/CHALARONNE
4,84,Auvergne-Rhône-Alpes,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE M...,MEXIMIEUX


## 2.3. Résolution des départements Corses (`2A`/`2B`) via `geo.api.gouv.fr` et normalisation finale (`cleaning_depistages.ipynb`)

**Ce que fait la cellule ci-dessous :**  
Interroge l'API officielle `https://geo.api.gouv.fr/communes` pour attribuer `2A` (Corse-du-Sud) ou `2B` (Haute-Corse) aux communes corses sans homonymes, applique la correction spécifique pour `L'Île-Rousse` (`2B`), normalise le texte en Unicode `NFC` et `zfill(2)`, puis dédoublonne pour obtenir `df_radiologie_final`.

In [9]:
def normaliser_ville(valeur):
    if pd.isna(valeur):
        return ""
    texte = unicodedata.normalize("NFKD", str(valeur))
    texte = "".join(c for c in texte if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", texte.lower())


url = "https://geo.api.gouv.fr/communes?fields=nom,codeDepartement"
try:
    resp_geo = requests.get(url, timeout=30)
    resp_geo.raise_for_status()
    communes = pd.DataFrame(resp_geo.json())
    communes["ville_normalisee"] = communes["nom"].map(normaliser_ville)
    communes_uniques = communes.loc[~communes["ville_normalisee"].duplicated(keep=False)]
    correspondance_corse = (
        communes_uniques.loc[communes_uniques["codeDepartement"].isin(["2A", "2B"])]
        .set_index("ville_normalisee")["codeDepartement"]
        .to_dict()
    )
except requests.exceptions.RequestException as exc:
    print(f"API geo.api.gouv.fr indisponible ({type(exc).__name__}), utilisation du référentiel Corse local.")
    correspondance_corse = {
        "ajaccio": "2A",
        "portovecchio": "2A",
        "propriano": "2A",
        "bastia": "2B",
        "borgo": "2B",
        "corte": "2B",
        "prunellidifiumorbo": "2B",
    }

departement_deduit = df_radiologie["ville"].map(normaliser_ville).map(correspondance_corse)
a_corriger = departement_deduit.notna()
df_radiologie.loc[a_corriger, "code_departement"] = departement_deduit.loc[a_corriger]

# Correction spécifique de L'Île-Rousse (Haute-Corse 2B)
masque_ile_rousse = df_radiologie["ville"].map(normaliser_ville).isin(["ilerousse", "lilerousse"])
df_radiologie.loc[masque_ile_rousse, ["Code_region", "Region", "code_departement", "departement"]] = [
    "94", "Corse", "2B", "Haute-Corse"
]

df_radiologie_final = df_radiologie[["Code_region", "Region", "code_departement", "departement", "nom_centre", "ville"]].copy()
for col in df_radiologie_final.columns:
    df_radiologie_final[col] = (
        df_radiologie_final[col]
        .astype("string")
        .map(lambda x: unicodedata.normalize("NFC", x) if pd.notna(x) else pd.NA)
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

for col in ["Code_region", "code_departement"]:
    df_radiologie_final[col] = (
        df_radiologie_final[col].str.upper().str.replace(r"\.0+$", "", regex=True).str.zfill(2)
    )

noms_regions_insee = {code: nom for code, (nom, _) in regions.items()}
df_radiologie_final["Region"] = df_radiologie_final["Code_region"].map(noms_regions_insee).astype("string").fillna(df_radiologie_final["Region"])
for code, nom in {"2A": "Corse-du-Sud", "2B": "Haute-Corse"}.items():
    masque = df_radiologie_final["code_departement"].eq(code).fillna(False)
    df_radiologie_final.loc[masque, ["departement", "Code_region", "Region"]] = [nom, "94", "Corse"]

df_radiologie_final = (
    df_radiologie_final.drop_duplicates().sort_values(["Region", "departement", "ville", "nom_centre"]).reset_index(drop=True)
)

print(f"df_radiologie_final prêt : {len(df_radiologie_final):,} lignes, {df_radiologie_final.duplicated().sum()} doublons")
display(df_radiologie_final.loc[df_radiologie_final["Code_region"] == "94"].head(10))

df_radiologie_final prêt : 1,571 lignes, 0 doublons


,Code_region,Region,code_departement,departement,nom_centre,ville
354,94,Corse,2A,Corse-du-Sud,CENTRE D'IMAGERIE DE LA FEMME - ESPACE MEDICAL...,AJACCIO
355,94,Corse,2A,Corse-du-Sud,CENTRE D'IMAGERIE MEDICALE DU CASONE,AJACCIO
356,94,Corse,2A,Corse-du-Sud,CABINET DE RADIOLOGIE LES ALBIZZIAS,PORTO VECCHIO
357,94,Corse,2A,Corse-du-Sud,SERVICE D'IMAGERIE MEDICALE POLYCLINIQUE DU SU...,PORTO-VECCHIO
358,94,Corse,2A,Corse-du-Sud,CENTRE D'IMAGERIE MEDICALE TIMAVI,PROPRIANO
359,94,Corse,2B,Haute-Corse,CABINET DU CENTRE - GROUPEMENT D'IMAGERIE MEDI...,BASTIA
360,94,Corse,2B,Haute-Corse,SERVICE D'IMAGERIE MÉDICALE - CENTRE HOSPITALI...,BASTIA
361,94,Corse,2B,Haute-Corse,CABINET D'IMAGERIE MEDICALE BORGO SELARL IMBM,BORGO
362,94,Corse,2B,Haute-Corse,SERVICE D'IMAGERIE MEDICALE - CENTRE HOSPITALI...,CORTE
363,94,Corse,2B,Haute-Corse,CABINET DE RADIOLOGIE - IMAGERIE MEDICALE DE L...,ILE ROUSSE


## 2.4. Chargement dans BigQuery (`Data_brute.df_radiologie_final`) avec schéma documenté & inventaire du Data Lake

**Ce que fait la cellule ci-dessous :**  
Définit le schéma BigQuery explicite (`bigquery.SchemaField` avec descriptions métier pour chaque colonne) utilisé lors du chargement de `df_radiologie_final` dans `Data_brute` (`WriteDisposition.WRITE_EMPTY`), vérifie sa présence dans BigQuery et liste les datasets consolidés du projet.

In [10]:
table_id = f"{PROJECT_ID}.Data_brute.df_radiologie_final"

schema = [
    bigquery.SchemaField("Code_region", "STRING", description="Code officiel INSEE de la région, conservé en texte pour préserver les zéros initiaux."),
    bigquery.SchemaField("Region", "STRING", description="Nom de la région administrative où se situe le cabinet de radiologie."),
    bigquery.SchemaField("code_departement", "STRING", description="Code officiel INSEE du département : deux caractères en métropole (dont 2A et 2B), ou trois chiffres en outre-mer."),
    bigquery.SchemaField("departement", "STRING", description="Nom du département où se situe le cabinet de radiologie."),
    bigquery.SchemaField("nom_centre", "STRING", description="Nom du cabinet ou du centre de radiologie, tel que renseigné dans la source."),
    bigquery.SchemaField("ville", "STRING", description="Nom de la ville où se situe le cabinet ou le centre de radiologie."),
]

job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition=bigquery.WriteDisposition.WRITE_EMPTY,
)

# Exécution sécurisée : la table étant déjà chargée en production, on ne réécrit que si elle est absente
existing_tables = {t.table_id for t in client.list_tables(f"{PROJECT_ID}.Data_brute")}
if "df_radiologie_final" not in existing_tables:
    job = client.load_table_from_dataframe(df_radiologie_final, table_id, job_config=job_config)
    job.result()

table = client.get_table(table_id)
print(f"Table BigQuery vérifiée : {table.full_table_id} ({table.num_rows:,} lignes)")
print("\n=== Architecture des datasets BigQuery ===")
for ds in client.list_datasets():
    nb_obj = len(list(client.list_tables(f"{PROJECT_ID}.{ds.dataset_id}")))
    print(f"- {ds.dataset_id:20s} : {nb_obj:2d} objets")

Table BigQuery vérifiée : cancer-du-sein-513216:Data_brute.df_radiologie_final (1,571 lignes)

=== Architecture des datasets BigQuery ===
- Data_brute           : 53 objets
- Data_clean           : 18 objets
- int_projet_cancer    :  8 objets
- mart_projet_cancer   :  9 objets
- stg_projet_cancer    : 26 objets


## 3.1. Couche Bronze / Staging dbt : inspection SQL (`stg_centres_radiologie.sql` & `stg_mortalite_region.sql`)

**Ce que fait la cellule ci-dessous :**  
Affiche deux modèles SQL représentatifs de la couche Staging (`stg_projet_cancer`) montrant l'harmonisation des libellés régionaux, le formatage `LPAD(..., 2, '0')`, la déduplication `QUALIFY ROW_NUMBER()` et la génération des clés primaires `SHA-256`.

In [11]:
for fname in ["stg_centres_radiologie.sql", "stg_mortalite_region.sql"]:
    sql_path = DBT_PROJECT_DIR / "models" / "staging" / fname
    print(f"=== {fname} ===")
    print(sql_path.read_text(encoding="utf-8"))
    print()

=== stg_centres_radiologie.sql ===
{{ config(materialized='view') }}

with source as (

    select * from {{ source('data_brute', 'df_radiologie_final') }}

),

cleaned as (

    select
        lpad(trim(cast(Code_region as string)), 2, '0') as code_region,
        case
            when trim(Region) = 'Auvergne et Rhône-Alpes' then 'Auvergne-Rhône-Alpes'
            when trim(Region) = 'Bourgogne et Franche-Comté' then 'Bourgogne-Franche-Comté'
            when trim(Region) = 'Nouvelle Aquitaine' then 'Nouvelle-Aquitaine'
            else trim(Region)
        end as nom_region,
        lpad(trim(cast(code_departement as string)), 2, '0') as code_departement,
        trim(departement) as nom_departement,
        trim(nom_centre) as nom_centre_radiologie,
        trim(ville) as ville

    from source
    qualify row_number() over (
        partition by trim(nom_centre), trim(ville), trim(code_departement)
        order by trim(nom_centre)
    ) = 1

)

select
    to_hex(sha256(to_json_st

## 3.2. Couche Bronze / Staging dbt : déploiement des 26 vues (`dbt run --select staging`)

**Ce que fait la cellule ci-dessous :**  
Exécute `dbt run --select staging` depuis Python pour matérialiser les **26 vues** du dataset `stg_projet_cancer` dans BigQuery à partir des 53 tables de `Data_brute`.

In [12]:
cmd = [str(DBT_EXE), "run", "--select", "staging", "--project-dir", str(DBT_PROJECT_DIR)]
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)

15:27:29  Running with dbt=1.12.6
15:27:30  Registered adapter: bigquery=1.12.2
15:27:31  Found 42 models, 371 data tests, 52 sources, 562 macros
15:27:31  
15:27:31  Concurrency: 4 threads (target='dev')
15:27:31  
15:27:32  4 of 26 START sql view model stg_projet_cancer.stg_depenses_onco ............... [RUN]
15:27:32  3 of 26 START sql view model stg_projet_cancer.stg_charge_region ............... [RUN]
15:27:32  1 of 26 START sql view model stg_projet_cancer.stg_centres_radiologie .......... [RUN]
15:27:32  2 of 26 START sql view model stg_projet_cancer.stg_charge_france ............... [RUN]
15:27:35  4 of 26 OK created sql view model stg_projet_cancer.stg_depenses_onco .......... [CREATE VIEW (0 processed) in 2.81s]
15:27:35  5 of 26 START sql view model stg_projet_cancer.stg_depenses_patho .............. [RUN]
15:27:35  2 of 26 OK created sql view model stg_projet_cancer.stg_charge_france .......... [CREATE VIEW (0 processed) in 2.86s]
15:27:35  6 of 26 START sql view model stg_

## 4.1. Couche Silver / Intermediate dbt : les 8 tables métier (`int_projet_cancer`) & analyse `< 50 ans`

**Ce que fait la cellule ci-dessous :**  
Liste les **8 tables** matérialisées dans `int_projet_cancer` avec leur volumétrie et interroge `int_incidence_age_depistage` pour illustrer l'augmentation de l'incidence depuis 1990 chez les femmes de 20 à 49 ans (hors dépistage organisé).

In [13]:
silver_tables = sorted(t.table_id for t in client.list_tables(f"{PROJECT_ID}.int_projet_cancer"))
print(f"Tables Silver dans int_projet_cancer ({len(silver_tables)}) :")
for t_name in silver_tables:
    t_obj = client.get_table(f"{PROJECT_ID}.int_projet_cancer.{t_name}")
    print(f"- {t_name:40s} ({t_obj.num_rows:,} lignes)")

query_silver = f"""
SELECT
    groupe_depistage,
    MIN(annee) AS annee_debut,
    MAX(annee) AS annee_fin,
    MAX(CASE WHEN annee = 1990 THEN ROUND(taux_incidence_brut, 1) END) AS taux_1990_pour_100k,
    MAX(CASE WHEN annee = 2023 THEN ROUND(taux_incidence_brut, 1) END) AS taux_2023_pour_100k,
    MAX(CASE WHEN annee = 2023 THEN ROUND(evolution_pct_depuis_1990, 1) END) AS evolution_pct_1990_2023
FROM `{PROJECT_ID}.int_projet_cancer.int_incidence_age_depistage`
WHERE niveau_agregation = 'Groupe dépistage'
GROUP BY groupe_depistage
ORDER BY groupe_depistage
"""
display(client.query(query_silver).to_dataframe())

C:\dev\dbt-workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Tables Silver dans int_projet_cancer (8) :
- int_carte_centres_depistage_ville        (9,006 lignes)
- int_charge_hospitaliere_sein             (126 lignes)
- int_comparatif_sein_vs_tous_cancers      (2,700 lignes)
- int_depenses_regionales_sein             (378 lignes)
- int_impact_depistage_territoires         (162 lignes)
- int_incidence_age_depistage              (612 lignes)
- int_mortalite_territoriale_sein          (198 lignes)
- int_nb_centre_taux_depistage_region      (126 lignes)


,groupe_depistage,annee_debut,annee_fin,taux_1990_pour_100k,taux_2023_pour_100k,evolution_pct_1990_2023
0,20-49 ans (hors dépistage),1990,2023,63.2,98.5,55.7
1,50-74 ans (cible dépistage),1990,2023,237.4,331.9,39.8
2,75 ans et plus (post dépistage),1990,2023,239.3,356.2,48.8


## 5.1. Couche Gold / Marts dbt : modèle en étoile Power BI (`mart_projet_cancer`)

**Ce que fait la cellule ci-dessous :**  
Interroge les tables finales prêtes pour Power BI : `mart_carte_centres_ville` (vérification du géocodage DROM `SAINT-JOSEPH, Réunion` sans duplication temporelle) et `mart_depistage_centres_region` jointe à `dim_regions` (période `2014–2022`).

In [14]:
query_gold_ville = f"""
SELECT code_region, nom_region, ville, localisation_complete, nb_centres
FROM `{PROJECT_ID}.mart_projet_cancer.mart_carte_centres_ville`
WHERE code_region = '04'
ORDER BY nb_centres DESC, ville
LIMIT 10
"""
display(client.query(query_gold_ville).to_dataframe())

query_gold_region = f"""
SELECT
    d.zone_geographique,
    m.nom_region,
    m.annee,
    m.nb_centres_region,
    m.taux_participation_std_pct,
    m.densite_population_hab_km2,
    m.nb_centres_pour_100k_femmes_50_74
FROM `{PROJECT_ID}.mart_projet_cancer.mart_depistage_centres_region` m
JOIN `{PROJECT_ID}.mart_projet_cancer.dim_regions` d
  ON m.code_region = d.code_region
WHERE m.est_derniere_annee = TRUE
ORDER BY m.taux_participation_std_pct DESC
"""
display(client.query(query_gold_region).to_dataframe())

C:\dev\dbt-workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,code_region,nom_region,ville,localisation_complete,nb_centres
0,04,La Réunion,ST PIERRE,"SAINT-PIERRE, Réunion",2
1,04,La Réunion,LE TAMPON,"LE TAMPON, Réunion",1
2,04,La Réunion,SAINT ANDRE,"SAINT ANDRE, Réunion",1
3,04,La Réunion,SAINT DENIS,"SAINT DENIS, Réunion",1
4,04,La Réunion,SAINT LOUIS,"SAINT LOUIS, Réunion",1
5,04,La Réunion,SAINT-PAUL,"SAINT-PAUL, Réunion",1
6,04,La Réunion,ST ANDRE,"SAINT-ANDRE, Réunion",1
7,04,La Réunion,ST BENOIT,"SAINT-BENOIT, Réunion",1
8,04,La Réunion,ST DENIS,"SAINT-DENIS, Réunion",1
9,04,La Réunion,ST GILLES LES HAUTS,"SAINT-GILLES LES HAUTS, Réunion",1


C:\dev\dbt-workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,zone_geographique,nom_region,annee,nb_centres_region,taux_participation_std_pct,densite_population_hab_km2,nb_centres_pour_100k_femmes_50_74
0,Hexagone,Pays de la Loire,2022,61,53.1,120.9,9.96
1,Hexagone,Bretagne,2022,64,52.4,125.8,11.13
2,Hexagone,Normandie,2022,73,51.9,111.6,13.10
3,Hexagone,Bourgogne-Franche-Comté,2022,61,51.8,58.7,12.79
4,Hexagone,Auvergne-Rhône-Alpes,2022,178,50.4,117.1,14.13
5,Hexagone,Centre-Val-de-Loire,2022,51,49.4,65.9,11.81
6,Hexagone,Grand Est,2022,137,48.2,96.8,14.95
7,Hexagone,Nouvelle-Aquitaine,2022,134,46.0,72.7,12.58
8,Hexagone,Hauts-de-France,2022,135,44.8,188.6,14.50
9,Outre-mer,Martinique,2022,9,43.2,320.1,12.01


## 5.2. Table Gold `mart_correlation_depistage_prevalence_mortalite_region` (Dépistage, Prévalence & Mortalité, 2015–2022)

**Ce que fait la cellule ci-dessous :**  
Interroge `mart_correlation_depistage_prevalence_mortalite_region` (144 lignes : 8 ans × 18 régions) jointe à `dim_regions` pour relier sur 2022 participation au dépistage, prévalences Ameli (active, sous surveillance, totale) et mortalité par groupe d'âge.

In [15]:
query_corr_2022 = f"""
SELECT
    r.libelle_region AS nom_region,
    r.zone_geographique,
    m.taux_participation_std_pct,
    m.prevalence_active_pct,
    m.prevalence_surveillance_pct,
    m.prevalence_totale_pct,
    m.taux_brut_mortalite_sein_pour_100k,
    m.nb_deces_sein_femmes,
    m.nb_deces_sein_moins_50_ans,
    m.nb_deces_sein_50_74_ans,
    m.nb_deces_sein_75_ans_plus
FROM `{PROJECT_ID}.mart_projet_cancer.mart_correlation_depistage_prevalence_mortalite_region` m
JOIN `{PROJECT_ID}.mart_projet_cancer.dim_regions` r
  ON m.code_region = r.code_region
WHERE m.annee = 2022
ORDER BY m.taux_participation_std_pct DESC
"""
display(client.query(query_corr_2022).to_dataframe())

C:\dev\dbt-workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,nom_region,zone_geographique,taux_participation_std_pct,prevalence_active_pct,prevalence_surveillance_pct,prevalence_totale_pct,taux_brut_mortalite_sein_pour_100k,nb_deces_sein_femmes,nb_deces_sein_moins_50_ans,nb_deces_sein_50_74_ans,nb_deces_sein_75_ans_plus
0,Pays de la Loire,Hexagone,53.1,0.734,1.335,2.069,37.33,741,67.34,280.66,393.0
1,Bretagne,Hexagone,52.4,0.736,1.313,2.049,38.25,672,44.43,258.57,369.0
2,Normandie,Hexagone,51.9,0.756,1.504,2.260,47.02,810,66.58,338.42,405.0
3,Bourgogne-Franche-Comté,Hexagone,51.8,0.758,1.452,2.210,39.50,567,37.23,207.77,322.0
4,Auvergne-Rhône-Alpes,Hexagone,50.4,0.695,1.269,1.964,33.79,1414,101.93,510.07,802.0
5,Centre-Val-de-Loire,Hexagone,49.4,0.721,1.362,2.083,44.19,587,38.83,232.17,316.0
6,Grand Est,Hexagone,48.2,0.735,1.525,2.260,38.33,1091,104.21,453.79,533.0
7,Nouvelle-Aquitaine,Hexagone,46.0,0.774,1.517,2.291,41.56,1315,90.54,510.46,714.0
8,Hauts-de-France,Hexagone,44.8,0.714,1.356,2.070,40.66,1256,101.68,533.32,621.0
9,Martinique,Outre-mer,43.2,0.549,0.837,1.386,28.01,55,7.73,30.27,17.0


## 5.3. Dimension `dim_tranches_age` & table `mart_correlation_age_region_sein` (Dépistage, Prévalence & Mortalité par âge, 2015–2022)

**Ce que fait la cellule ci-dessous :**  
Affiche les 15 tranches d'âge de `dim_tranches_age` (seuil strict à 50 ans), puis agrège sur 2022 `mart_correlation_age_region_sein` par `groupe_depistage` pour comparer prévalence active/totale, participation et décès par âge.

In [16]:
display(
    client.query(
        f"SELECT ordre_tranche_age, code_tranche_age_5, libelle_tranche_age_5, groupe_depistage, est_cible_depistage FROM `{PROJECT_ID}.mart_projet_cancer.dim_tranches_age` ORDER BY ordre_tranche_age"
    ).to_dataframe()
)

query_corr_age_2022 = f"""
SELECT
    a.ordre_groupe_depistage,
    a.groupe_depistage,
    a.est_cible_depistage,
    SUM(m.population_femmes) AS population_femmes_total,
    ROUND(AVG(m.taux_participation_brut_tranche_pct), 2) AS taux_participation_moyen_pct,
    ROUND(AVG(m.prevalence_active_pct), 3) AS prevalence_active_moy_pct,
    ROUND(AVG(m.prevalence_totale_pct), 3) AS prevalence_totale_moy_pct,
    ROUND(SUM(m.nb_deces_sein_femmes), 0) AS nb_deces_sein
FROM `{PROJECT_ID}.mart_projet_cancer.mart_correlation_age_region_sein` m
JOIN `{PROJECT_ID}.mart_projet_cancer.dim_tranches_age` a
  ON m.code_tranche_age_5 = a.code_tranche_age_5
WHERE m.annee = 2022
GROUP BY 1, 2, 3
ORDER BY 1
"""
display(client.query(query_corr_age_2022).to_dataframe())

C:\dev\dbt-workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ordre_tranche_age,code_tranche_age_5,libelle_tranche_age_5,groupe_depistage,est_cible_depistage
0,1,25-29,25-29 ans,Moins de 50 ans (hors dépistage),False
1,2,30-34,30-34 ans,Moins de 50 ans (hors dépistage),False
2,3,35-39,35-39 ans,Moins de 50 ans (hors dépistage),False
3,4,40-44,40-44 ans,Moins de 50 ans (hors dépistage),False
4,5,45-49,45-49 ans,Moins de 50 ans (hors dépistage),False
5,6,50-54,50-54 ans,50-74 ans (cible dépistage),True
6,7,55-59,55-59 ans,50-74 ans (cible dépistage),True
7,8,60-64,60-64 ans,50-74 ans (cible dépistage),True
8,9,65-69,65-69 ans,50-74 ans (cible dépistage),True
9,10,70-74,70-74 ans,50-74 ans (cible dépistage),True


C:\dev\dbt-workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,ordre_groupe_depistage,groupe_depistage,est_cible_depistage,population_femmes_total,taux_participation_moyen_pct,prevalence_active_moy_pct,prevalence_totale_moy_pct,nb_deces_sein
0,1,Moins de 50 ans (hors dépistage),False,10559551,NaN,0.387,0.667,1063.0
1,2,50-74 ans (cible dépistage),True,10805545,45.96,1.278,3.382,5029.0
2,3,75 ans et plus (post-dépistage),False,3978812,NaN,1.215,4.947,6664.0


## 5.4. Table Gold `mart_oncologie_depenses_charge_region` (Budget Oncologie & Charge Hospitalière sur 10 spécialités, 2018–2024)

**Ce que fait la cellule ci-dessous :**  
Interroge `mart_oncologie_depenses_charge_region` (1 260 lignes : 7 ans × 18 régions × 10 spécialités détaillées Ameli ↔ DREES) sur l'année 2022 pour restituer la part du cancer du sein face aux autres cancers dans le budget dédié à l'oncologie et dans la charge hospitalière MCO (données des 2 Donuts Power BI).

In [17]:
query_onco_2022 = f"""
SELECT
    ordre_specialite,
    specialite_oncologique,
    SUM(nb_patients_ameli) AS patients_ameli,
    ROUND(SUM(depense_totale_euros) / 1e6, 2) AS budget_M_euros,
    ROUND(100 * SUM(depense_totale_euros) / SUM(SUM(depense_totale_euros)) OVER(), 2) AS part_budget_pct,
    SUM(nb_sejours_total) AS sejours_mco,
    ROUND(100 * SUM(nb_sejours_total) / SUM(SUM(nb_sejours_total)) OVER(), 2) AS part_sejours_pct,
    ROUND(SUM(nb_journees_estimees), 0) AS journees_estimees,
    ROUND(100 * SUM(nb_journees_estimees) / SUM(SUM(nb_journees_estimees)) OVER(), 2) AS part_journees_pct
FROM `{PROJECT_ID}.mart_projet_cancer.mart_oncologie_depenses_charge_region`
WHERE annee = 2022
GROUP BY 1, 2
ORDER BY 1
"""
df_onco_2022 = client.query(query_onco_2022).to_dataframe()
tbl_onco = client.get_table(f"{PROJECT_ID}.mart_projet_cancer.mart_oncologie_depenses_charge_region")
print(f"Volumétrie mart_oncologie_depenses_charge_region : {tbl_onco.num_rows:,} lignes")
display(df_onco_2022)

C:\dev\dbt-workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Volumétrie mart_oncologie_depenses_charge_region : 1,260 lignes


,ordre_specialite,specialite_oncologique,patients_ameli,budget_M_euros,part_budget_pct,sejours_mco,part_sejours_pct,journees_estimees,part_journees_pct
0,1,Cancer du sein de la femme,724940,4171.67,16.96,77602,10.54,154442.0,4.07
1,2,Cancer bronchopulmonaire,168910,3287.09,13.36,56344,7.65,425314.0,11.21
2,3,Cancer de la prostate,553580,2511.39,10.21,62634,8.50,164247.0,4.33
3,4,Cancer colorectal,367120,1814.90,7.38,70517,9.58,489750.0,12.91
4,5,Métastases (secondaires),324708,2141.43,8.70,78344,10.64,538815.0,14.20
5,6,"Autres digestifs (pancréas, foie, estomac, œsophage)",251927,1662.02,6.76,60901,8.27,475184.0,12.52
6,7,Cancer de la vessie,223032,1471.07,5.98,53899,7.32,245345.0,6.47
7,8,Hématologie (leucémies) & SNC,93255,615.46,2.50,22571,3.06,324710.0,8.56
8,9,"ORL, Utérus, Peau & Thyroïde",262085,1729.38,7.03,63337,8.60,263238.0,6.94
9,10,Autres cancers (non détaillés),787838,5198.34,21.13,190289,25.84,713716.0,18.81


## 5.5. Table Gold `mart_comparatif_sein_vs_onco` (Cancer du sein vs Tous cancers par région et tranche d'âge, 2015–2024)

**Ce que fait la cellule ci-dessous :**  
Interroge `mart_comparatif_sein_vs_onco` (2 700 lignes : 10 ans × 18 régions × 15 tranches d'âge quinquennales dès 25 ans) jointe à `dim_tranches_age` sur l'année 2022 pour comparer au niveau national par tranche d'âge le **Cancer du sein** face à **Tous cancers** (périmètres Femmes uniquement et Tous sexes) : décès CépiDc, taux pour 100 000, patients actifs Ameli et ratio `décès / patients actifs (%)`.

In [18]:
query_comp_2022 = f"""
SELECT
    a.ordre_tranche_age,
    m.tranche_age,
    ROUND(SUM(m.nb_deces_sein), 0) AS deces_sein,
    ROUND(SUM(m.nb_deces_tous_cancers_femmes), 0) AS deces_tous_cancers_f,
    ROUND(SUM(m.nb_deces_tous_cancers), 0) AS deces_tous_cancers_tot,
    ROUND(SAFE_DIVIDE(SUM(m.nb_deces_sein) * 100000.0, SUM(m.population_femmes)), 1) AS taux_sein_100k,
    ROUND(SAFE_DIVIDE(SUM(m.nb_deces_tous_cancers_femmes) * 100000.0, SUM(m.population_femmes)), 1) AS taux_tous_cancers_f_100k,
    SUM(m.nb_patients_actifs_sein) AS actifs_sein,
    SUM(m.nb_patients_actifs_tous_cancers_femmes) AS actifs_tous_cancers_f,
    SUM(m.nb_patients_actifs_tous_cancers) AS actifs_tous_cancers_tot,
    ROUND(SAFE_DIVIDE(SUM(m.nb_deces_sein) * 100.0, SUM(m.nb_patients_actifs_sein)), 2) AS ratio_deces_actifs_sein_pct,
    ROUND(SAFE_DIVIDE(SUM(m.nb_deces_tous_cancers_femmes) * 100.0, SUM(m.nb_patients_actifs_tous_cancers_femmes)), 2) AS ratio_deces_actifs_tous_cancers_f_pct,
    ROUND(SAFE_DIVIDE(SUM(m.nb_deces_tous_cancers) * 100.0, SUM(m.nb_patients_actifs_tous_cancers)), 2) AS ratio_deces_actifs_tous_cancers_tot_pct
FROM `{PROJECT_ID}.mart_projet_cancer.mart_comparatif_sein_vs_onco` m
JOIN `{PROJECT_ID}.mart_projet_cancer.dim_tranches_age` a
  ON m.tranche_age = a.code_tranche_age_5
WHERE m.annee = 2022
GROUP BY 1, 2
ORDER BY 1
"""
df_comp_2022 = client.query(query_comp_2022).to_dataframe()
tbl_comp = client.get_table(f"{PROJECT_ID}.mart_projet_cancer.mart_comparatif_sein_vs_onco")
dc_sein = df_comp_2022['deces_sein'].sum()
dc_f = df_comp_2022['deces_tous_cancers_f'].sum()
dc_tot = df_comp_2022['deces_tous_cancers_tot'].sum()
act_sein = df_comp_2022['actifs_sein'].sum()
act_f = df_comp_2022['actifs_tous_cancers_f'].sum()
act_tot = df_comp_2022['actifs_tous_cancers_tot'].sum()
print(f"Volumétrie mart_comparatif_sein_vs_onco : {tbl_comp.num_rows:,} lignes")
print(f"France 2022 (25+) — Cancer du sein     : {dc_sein:,.0f} décès | {act_sein:,} patientes actives | ratio {dc_sein / act_sein * 100:.2f} %")
print(f"France 2022 (25+) — Tous cancers (F)   : {dc_f:,.0f} décès | {act_f:,} patientes actives | ratio {dc_f / act_f * 100:.2f} %")
print(f"France 2022 (25+) — Tous cancers (Tot) : {dc_tot:,.0f} décès | {act_tot:,} patients actifs   | ratio {dc_tot / act_tot * 100:.2f} %")
display(df_comp_2022)

C:\dev\dbt-workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Volumétrie mart_comparatif_sein_vs_onco : 2,700 lignes
France 2022 (25+) — Cancer du sein     : 12,756 décès | 249,610 patientes actives | ratio 5.11 %
France 2022 (25+) — Tous cancers (F)   : 72,395 décès | 792,420 patientes actives | ratio 9.14 %
France 2022 (25+) — Tous cancers (Tot) : 163,657 décès | 1,690,450 patients actifs   | ratio 9.68 %


,ordre_tranche_age,tranche_age,deces_sein,deces_tous_cancers_f,deces_tous_cancers_tot,taux_sein_100k,taux_tous_cancers_f_100k,actifs_sein,actifs_tous_cancers_f,actifs_tous_cancers_tot,ratio_deces_actifs_sein_pct,ratio_deces_actifs_tous_cancers_f_pct,ratio_deces_actifs_tous_cancers_tot_pct
0,1,25-29,39.0,166.0,293.0,2.1,8.7,630,6030,9870,6.26,2.75,2.97
1,2,30-34,44.0,183.0,320.0,2.1,8.7,2490,12060,17470,1.75,1.52,1.83
2,3,35-39,217.0,612.0,1111.0,9.9,28.1,6350,19660,27480,3.41,3.11,4.04
3,4,40-44,216.0,612.0,1120.0,9.9,28.1,12680,29120,40980,1.71,2.10,2.73
4,5,45-49,547.0,1927.0,4039.0,25.0,87.9,19380,39930,58110,2.83,4.83,6.95
5,6,50-54,567.0,1997.0,4172.0,24.9,87.9,26660,57620,92000,2.12,3.47,4.53
6,7,55-59,946.0,4914.0,12039.0,41.5,215.6,25570,66590,124240,3.70,7.38,9.69
7,8,60-64,906.0,4711.0,11365.0,41.7,216.7,28290,81500,172830,3.20,5.78,6.58
8,9,65-69,1327.0,8605.0,22420.0,63.9,414.5,30770,96110,225940,4.31,8.95,9.92
9,10,70-74,1283.0,8316.0,21391.0,64.0,414.8,34120,112010,277660,3.76,7.42,7.70


## 6.1. Tests de qualité (`dbt test --select marts`) et suivi GitHub

**Ce que fait la cellule ci-dessous :**  
Exécute la suite de **118 tests automatisés dbt** (`unique`, `not_null`, `accepted_values`, `relationships` vers `dim_regions` et `dim_tranches_age`) sur les 8 tables Gold dbt de `mart_projet_cancer` et affiche l'historique récent des commits du dépôt `Hythus/dbt-bigquery-analyse-cancer-du-sein`.

In [19]:
cmd_test = [str(DBT_EXE), "test", "--select", "marts", "--project-dir", str(DBT_PROJECT_DIR)]
res_test = subprocess.run(cmd_test, capture_output=True, text=True)
print(res_test.stdout)
if res_test.returncode != 0:
    print(res_test.stderr)

res_git = subprocess.run(
    ["git", "-C", str(DBT_PROJECT_DIR), "log", "-n", "5", "--oneline"],
    capture_output=True,
    text=True,
)
print("=== Derniers commits GitHub ===")
print(res_git.stdout)

15:27:58  Running with dbt=1.12.6
15:27:59  Registered adapter: bigquery=1.12.2
15:28:00  Found 42 models, 371 data tests, 52 sources, 562 macros
15:28:00  
15:28:00  Concurrency: 4 threads (target='dev')
15:28:00  
15:28:00  1 of 118 START test accepted_values_dim_regions_zone_geographique__Hexagone__Outre_mer  [RUN]
15:28:00  3 of 118 START test accepted_values_mart_oncologie_depenses_charge_region_specialite_oncologique__Cancer_du_sein_de_la_femme__Cancer_bronchopulmonaire__Cancer_de_la_prostate__Cancer_colorectal__M_tastases_secondaires___Autres_digestifs_pancr_as_foie_estomac_sophage___Cancer_de_la_vessie__H_matologie_leuc_mies_SNC__ORL_Ut_rus_Peau_Thyro_de__Autres_cancers_non_d_taill_s_  [RUN]
15:28:00  4 of 118 START test not_null_dim_regions_chef_lieu_region ...................... [RUN]
15:28:00  2 of 118 START test accepted_values_dim_tranches_age_groupe_depistage__Moins_de_50_ans_hors_d_pistage___50_74_ans_cible_d_pistage___75_ans_et_plus_post_d_pistage_  [RUN]
15:28:02  4 of

## 7.1. Restitution Power BI : reproduction en Pandas des mesures DAX sur l'année 2022

**Ce que fait la cellule ci-dessous :**  
Reproduit en Python les indicateurs DAX implémentés dans `_Mesures` pour Power BI Desktop sur 2022 : offre de centres, participation au dépistage, prévalence active/totale Ameli et mortalité (`mart_depistage_centres_region` × `mart_correlation_depistage_prevalence_mortalite_region`).

In [20]:
query_kpi = f"""
SELECT
    c.nom_region,
    c.nb_centres_region,
    c.femmes_50_74,
    c.nb_femmes_participantes_estime,
    c.densite_population_hab_km2,
    c.taux_participation_std_pct,
    c.nb_centres_pour_100k_femmes_50_74,
    corr.prevalence_active_pct,
    corr.prevalence_totale_pct,
    corr.taux_brut_mortalite_sein_pour_100k,
    corr.nb_patientes_actives_sein,
    corr.nb_deces_sein_femmes
FROM `{PROJECT_ID}.mart_projet_cancer.mart_depistage_centres_region` c
JOIN `{PROJECT_ID}.mart_projet_cancer.mart_correlation_depistage_prevalence_mortalite_region` corr
  ON c.annee = corr.annee AND c.code_region = corr.code_region
WHERE c.annee = 2022
"""

df_pbi = client.query(query_kpi).to_dataframe()
taux_france_2022 = (df_pbi["nb_femmes_participantes_estime"].sum() / df_pbi["femmes_50_74"].sum()) * 100

df_pbi["ecart_vs_france_pts"] = (df_pbi["taux_participation_std_pct"] - taux_france_2022).round(2)
df_pbi["letalite_apparente_pct"] = ((df_pbi["nb_deces_sein_femmes"] / df_pbi["nb_patientes_actives_sein"]) * 100).round(1)
df_pbi["profil_densite"] = pd.cut(
    df_pbi["densite_population_hab_km2"],
    bins=[-1, 80, 150, float("inf")],
    labels=["Faible (< 80 hab/km²)", "Intermédiaire (80-150)", "Forte (> 150 hab/km²)"],
)

print(f"Taux de participation national pondéré (2022) : {taux_france_2022:.2f} %")
print(f"Total centres recensés (hors Mayotte)        : {df_pbi['nb_centres_region'].sum():,}")
display(
    df_pbi[
        [
            "nom_region", "profil_densite", "nb_centres_pour_100k_femmes_50_74",
            "taux_participation_std_pct", "ecart_vs_france_pts",
            "prevalence_active_pct", "prevalence_totale_pct",
            "taux_brut_mortalite_sein_pour_100k", "letalite_apparente_pct",
        ]
    ].sort_values("taux_participation_std_pct", ascending=False)
)

C:\dev\dbt-workspace\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2131: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Taux de participation national pondéré (2022) : 44.82 %
Total centres recensés (hors Mayotte)        : 1,571


,nom_region,profil_densite,nb_centres_pour_100k_femmes_50_74,taux_participation_std_pct,ecart_vs_france_pts,prevalence_active_pct,prevalence_totale_pct,taux_brut_mortalite_sein_pour_100k,letalite_apparente_pct
6,Pays de la Loire,Intermédiaire (80-150),9.96,53.1,8.28,0.734,2.069,37.33,5.0
0,Bretagne,Intermédiaire (80-150),11.13,52.4,7.58,0.736,2.049,38.25,5.1
15,Normandie,Intermédiaire (80-150),13.10,51.9,7.08,0.756,2.260,47.02,6.2
10,Bourgogne-Franche-Comté,Faible (< 80 hab/km²),12.79,51.8,6.98,0.758,2.210,39.50,5.2
3,Auvergne-Rhône-Alpes,Intermédiaire (80-150),14.13,50.4,5.58,0.695,1.964,33.79,4.8
4,Centre-Val-de-Loire,Faible (< 80 hab/km²),11.81,49.4,4.58,0.721,2.083,44.19,6.1
5,Grand Est,Intermédiaire (80-150),14.95,48.2,3.38,0.735,2.260,38.33,5.4
16,Nouvelle-Aquitaine,Faible (< 80 hab/km²),12.58,46.0,1.18,0.774,2.291,41.56,5.3
1,Hauts-de-France,Forte (> 150 hab/km²),14.50,44.8,-0.02,0.714,2.070,40.66,5.7
7,Martinique,Forte (> 150 hab/km²),12.01,43.2,-1.62,0.549,1.386,28.01,4.9
